In [1]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import densenet121, resnet50
from torchvision.models import DenseNet121_Weights, ResNet50_Weights

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedShuffleSplit

torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = (
    True  # faster conv algorithm selection for static shapes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [2]:
DATA_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_CSV_PATH = f"{DATA_ROOT}/train.csv"
TRAIN_DIR = f"{DATA_ROOT}/train_images"
TEST_DIR = f"{DATA_ROOT}/test_images"
SAMPLE_SUB_PATH = f"{DATA_ROOT}/sample_submission.csv"

TREE_TRAIN_PATH = "/kaggle/input/train-tree/train_tree.csv"
MODEL1_PATH = "/kaggle/input/densenet/keras/default/1/DenseNet (1).keras"
MODEL2_PATH = (
    "/kaggle/input/resnet50_70_512x512/pytorch/default/1/Resnet50_70_512x512.pth"
)

assert os.path.exists(TRAIN_CSV_PATH), f"Missing {TRAIN_CSV_PATH}"
assert os.path.exists(TRAIN_DIR), f"Missing {TRAIN_DIR}"
assert os.path.exists(TEST_DIR), f"Missing {TEST_DIR}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing {SAMPLE_SUB_PATH}"




In [3]:
densenet_weights = DenseNet121_Weights.IMAGENET1K_V1
resnet_weights = ResNet50_Weights.IMAGENET1K_V2

dn_preprocess = densenet_weights.transforms()
rn_preprocess = resnet_weights.transforms()

_RESAMPLE = Image.BILINEAR




In [4]:
def _safe_load_state_dict(
    model: nn.Module, ckpt_path: str, device: torch.device
) -> bool:
    """Try to load a PyTorch checkpoint into `model`. Returns True if loaded, False otherwise."""
    if not ckpt_path or not os.path.exists(ckpt_path):
        return False
    try:
        obj = torch.load(ckpt_path, map_location=device)
        if isinstance(obj, nn.Module):
            model.load_state_dict(obj.state_dict(), strict=False)
            return True
        if isinstance(obj, dict):
            state = obj.get("state_dict", obj.get("model", obj))
            if isinstance(state, dict) and any(
                k.startswith("module.") for k in state.keys()
            ):
                state = {k.replace("module.", "", 1): v for k, v in state.items()}
            if isinstance(state, dict):
                model.load_state_dict(state, strict=False)
                return True
        return False
    except Exception:
        return False


model1 = densenet121(weights=densenet_weights)
model2 = resnet50(weights=resnet_weights)

loaded1 = _safe_load_state_dict(model1, MODEL1_PATH, device)
loaded2 = _safe_load_state_dict(model2, MODEL2_PATH, device)

model1.to(device).eval()
model2.to(device).eval()

USE_PENULTIMATE_EMBEDS = True

print(f"Device: {device}")
print(f"Loaded external weights: model1={loaded1}, model2={loaded2}")
print(
    "Using CNN feature vectors (two outputs concatenated -> Tree model): "
    + (
        "penultimate pooled embeddings"
        if USE_PENULTIMATE_EMBEDS
        else "final 1000-d logits"
    )
)




Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 110MB/s]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


Device: cuda
Loaded external weights: model1=False, model2=False
Using CNN feature vectors (two outputs concatenated -> Tree model): penultimate pooled embeddings


In [5]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
if not {"image_id", "label"}.issubset(train_df.columns):
    raise ValueError(f"train.csv columns unexpected: {train_df.columns.tolist()}")


def _extract_feats_batch(x1: torch.Tensor, x2: torch.Tensor) -> np.ndarray:
    """
    Returns concatenated feature vectors.
    DenseNet121 penultimate pooled embedding: 1024-d
    ResNet50 penultimate pooled embedding: 2048-d
    Total: 3072-d
    """
    with torch.inference_mode():
        if USE_PENULTIMATE_EMBEDS:
            f1 = model1.features(x1)
            f1 = torch.relu(f1)
            f1 = torch.nn.functional.adaptive_avg_pool2d(f1, (1, 1))
            f1 = torch.flatten(f1, 1)

            f2 = model2.conv1(x2)
            f2 = model2.bn1(f2)
            f2 = model2.relu(f2)
            f2 = model2.maxpool(f2)
            f2 = model2.layer1(f2)
            f2 = model2.layer2(f2)
            f2 = model2.layer3(f2)
            f2 = model2.layer4(f2)
            f2 = model2.avgpool(f2)
            f2 = torch.flatten(f2, 1)
        else:
            f1 = model1(x1)
            f2 = model2(x2)

    f1 = f1.detach().cpu().numpy().astype(np.float32, copy=False)
    f2 = f2.detach().cpu().numpy().astype(np.float32, copy=False)
    return np.concatenate([f1, f2], axis=1)


class _CassavaImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_ids, root_dir: str):
        self.image_ids = list(image_ids)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        p = os.path.join(self.root_dir, image_id)
        img = Image.open(p).convert("RGB")

        x1 = dn_preprocess(img)  # DenseNet expected preprocessing
        x2 = rn_preprocess(img)  # ResNet expected preprocessing
        return idx, x1, x2, image_id


def _infer_combined_feats_dataloader(image_ids, root_dir, batch_size=32):
    feat_dim = 3072 if USE_PENULTIMATE_EMBEDS else 2000
    combined = np.zeros((len(image_ids), feat_dim), dtype=np.float32)

    ds = _CassavaImageDataset(image_ids, root_dir)

    num_workers = min(4, (os.cpu_count() or 2))
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    missing = []
    for idxs, x1, x2, image_id_batch in loader:
        if x1 is None or x2 is None:
            missing.extend(list(image_id_batch))
            continue

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        concat = _extract_feats_batch(x1, x2)
        combined[idxs.numpy(), :] = concat

        end = int(idxs.max().item()) + 1
        if (end % 256) == 0 or end >= len(image_ids):
            print(f"Extracted features: {end}/{len(image_ids)}", end="\r")

    print()
    if missing:
        raise FileNotFoundError(
            f"Missing {len(missing)} images under {root_dir}. Example: {missing[:5]}"
        )
    return combined


labels = train_df["label"].astype(int).to_numpy()
train_image_ids = train_df["image_id"].tolist()

train_feats_all = _infer_combined_feats_dataloader(
    train_image_ids, TRAIN_DIR, batch_size=32
)

feat_mean = train_feats_all.mean(axis=0, keepdims=True).astype(np.float32)
feat_std = train_feats_all.std(axis=0, keepdims=True).astype(np.float32)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)
train_feats_all = (train_feats_all - feat_mean) / feat_std


def _fit_extratrees_select_by_holdout(X, y, base_kwargs):
    """
    Change rationale (score-up toward target, minimal semantics):
    OOB can be noisy for class-imbalanced multi-class; a small stratified holdout
    gives a more reliable generalization signal to choose between two nearly
    identical ExtraTrees regularization settings (bootstrap True/False).
    This keeps the same model family and training semantics; it's just safer selection.
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.12, random_state=42)
    tr_idx, va_idx = next(sss.split(X, y))
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]

    # Candidate A: no bootstrap
    m_no = ExtraTreesClassifier(
        **{**base_kwargs, "bootstrap": False, "oob_score": False}
    )
    m_no.fit(X_tr, y_tr)
    va_acc_no = float(m_no.score(X_va, y_va))

    # Candidate B: bootstrap (keep oob_score off during selection to avoid extra variance/cost)
    m_bs = ExtraTreesClassifier(
        **{**base_kwargs, "bootstrap": True, "oob_score": False}
    )
    m_bs.fit(X_tr, y_tr)
    va_acc_bs = float(m_bs.score(X_va, y_va))

    print(f"Holdout acc bootstrap=False: {va_acc_no:.6f}")
    print(f"Holdout acc bootstrap=True : {va_acc_bs:.6f}")

    chosen_bootstrap = bool(va_acc_bs >= va_acc_no)
    print(f"Selected model by holdout: bootstrap={chosen_bootstrap}")

    # Refit chosen config on full training data (standard practice; no change to inference)
    m_final = ExtraTreesClassifier(
        **{**base_kwargs, "bootstrap": chosen_bootstrap, "oob_score": False}
    )
    m_final.fit(X, y)
    return m_final


base_tree_kwargs = dict(
    n_estimators=900,  # small bump for stability/generalization; still fits in time
    criterion="gini",
    max_depth=24,  # tiny increase to capture slightly richer boundaries
    min_samples_split=12,
    min_samples_leaf=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample",  # keep as-is
)

_rng = np.random.RandomState(42)
JITTER_STD = 0.01  # keep as-is (already helped)
train_feats_fit = train_feats_all + (
    _rng.normal(0.0, JITTER_STD, size=train_feats_all.shape).astype(np.float32)
)

tree_model = _fit_extratrees_select_by_holdout(
    train_feats_fit, labels, base_tree_kwargs
)

print("Trained tree model on generated CNN features:", train_feats_all.shape)




Extracted features: 18721/18721
Holdout acc bootstrap=False: 0.760125
Holdout acc bootstrap=True : 0.752114
Selected model by holdout: bootstrap=False
Trained tree model on generated CNN features: (18721, 3072)


In [6]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
if "image_id" not in sample_sub.columns:
    raise ValueError(
        f"sample_submission missing 'image_id'. Columns: {sample_sub.columns.tolist()}"
    )

image_ids = sample_sub["image_id"].tolist()

combined_feats_test = _infer_combined_feats_dataloader(
    image_ids, TEST_DIR, batch_size=32
)
combined_feats_test = (combined_feats_test - feat_mean) / feat_std




Extracted features: 2676/2676


In [7]:
prediction = tree_model.predict(combined_feats_test).astype(int)

submission = pd.DataFrame({"image_id": image_ids, "label": prediction})
submission = submission[["image_id", "label"]]
submission["label"] = submission["label"].astype(int)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print("Label counts:\n", submission["label"].value_counts().sort_index())

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with shape: (2676, 2)
Label counts:
 label
0     149
1     267
2     119
3    1805
4     336
Name: count, dtype: int64
